In [1]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [3]:
# out_dir = f"./out/onemotif_twostates_pos/"
# out_dir = f"/data/cb/mihirb14/projects/BoltzDesign1/out/test/zinc/onemotif_twostates_pos"

out_dir = f"out/onemotif_twostates_neg_OQO/"
motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
motif_out_dir = os.path.join(out_dir,motif)

In [4]:


def motif_results(motif_out_dir, motif_pdb, motif):
    rows = []
    for design_dir in sorted(glob.glob(os.path.join(motif_out_dir, "design*"))):
        design_name = os.path.basename(design_dir)

        # load motif mask
        with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
            motif_mask = pickle.load(f)["motif_mask"]

        for state in [0, 1]:
            with open(os.path.join(design_dir, f"state{state}.pkl"), "rb") as f:
                outdict = pickle.load(f)

            for sample_idx in range(5):  # assume 5 samples per state
                pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
                if not os.path.exists(pdb_file):
                    continue

                rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)

                rows.append({
                    "design": design_name,
                    "state": state,
                    "sample": sample_idx,
                    "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                    "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                    "ptm": outdict["ptm"].cpu().numpy()[sample_idx],
                })
                
    full = pd.DataFrame(rows)

    agg = full.groupby(["design", "state"]).agg(
        motifRMSD_mean=("motifrmsd", "mean"),
        motifRMSD_std=("motifrmsd", "std"),
        plddt=("plddt", "mean"),
        ptm=("ptm", "mean")
    ).reset_index()

    agg = agg.pivot(index="design", columns="state").reset_index()
    agg.columns = ["_".join(map(str, col)).rstrip("_") for col in agg.columns.to_flat_index()]
    agg = agg.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))
    
    agg.to_csv(os.path.join(motif_out_dir,"_aggresults.csv"),index=False)
    full.to_csv(os.path.join(motif_out_dir,"_fullresults.csv"),index=False)


    return full, agg



df_full,df_agg = motif_results(motif_out_dir,motif_pdb, motif)
df_full

,design,state,sample,motifrmsd,plddt,ptm
0,design0,0,0,0.824371,0.792900,0.604228
1,design0,0,1,0.888581,0.782107,0.590002
2,design0,0,2,0.853081,0.786949,0.626292
3,design0,0,3,0.945256,0.789702,0.601309
4,design0,0,4,0.810755,0.794108,0.626148
...,...,...,...,...,...,...
495,design9,1,0,4.831772,0.637067,0.602008
496,design9,1,1,5.026365,0.618095,0.553604
497,design9,1,2,5.108874,0.629460,0.577872
498,design9,1,3,4.698315,0.632163,0.583528


In [6]:
len(df_agg[(df_agg["motifRMSD_mean_unbound"]>1.0) & (df_agg["motifRMSD_mean_bound"]<=1.0)]) # condition for one motif two states positive allostery

2

In [5]:


def plot_scatter_motifrmsd(df, motif):
    xrange = [0, 10]
    yrange = [0, 10]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("", "with std ± 1")
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=True,
                colorbar=dict(title="pLDDT",outlinewidth=0)
            ),
            text=df["design"],
            name="Designs"
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=df["motifRMSD_mean_unbound"],
            y=df["motifRMSD_mean_bound"],
            mode="markers",
            marker=dict(
                color=df["plddt_unbound"],
                colorscale="sunsetdark",
                showscale=False
            ),
            text=df["design"],
            error_x=dict(array=df["motifRMSD_std_unbound"], color="gray", thickness=1),
            error_y=dict(array=df["motifRMSD_std_bound"], color="gray", thickness=1),
            name="Designs (err)"
        ),
        row=1, col=2
    )

    for c in [1, 2]:
        fig.add_shape(
            type="line", x0=0, y0=0, x1=10, y1=10,
            line=dict(color="lightgray", dash="dash"),
            row=1, col=c
        )

    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=1)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=1)
    fig.update_xaxes(range=xrange, title="Unbound motif RMSD", row=1, col=2)
    fig.update_yaxes(range=yrange, title="Bound motif RMSD", row=1, col=2)

    fig.update_layout(
        width=1200, height=600,
        title=f"({motif}) Unbound vs Bound motif RMSD",
        yaxis_scaleanchor="x",
        yaxis2_scaleanchor="x2",
        showlegend=False
    )

    return fig


plot_scatter_motifrmsd(df_agg, motif)


### onemotif_twostates_neg with sm ligand modulation

In [6]:
successes_neg = []
out_dir = f"out/onemotif_twostates_neg_OQO/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0)  & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ]
    )
    
    successes_neg.append({
        "motif": motif,
        "effector":"OQO",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg = pd.DataFrame(successes_neg)
df_neg


,motif,effector,task,success_count
0,1bcf,OQO,onemotif_twostates_neg,0
1,1prw,OQO,onemotif_twostates_neg,5
2,1qjg,OQO,onemotif_twostates_neg,5
3,1ycr,OQO,onemotif_twostates_neg,19
4,2kl8,OQO,onemotif_twostates_neg,1
5,3ixt,OQO,onemotif_twostates_neg,13
6,4jhw,OQO,onemotif_twostates_neg,0
7,4zyp,OQO,onemotif_twostates_neg,3
8,5ius,OQO,onemotif_twostates_neg,0
9,5tpn,OQO,onemotif_twostates_neg,0


In [7]:
len(df_neg[df_neg["success_count"]!=0])

10

### plot all (onemotif_twostates_pos)

In [8]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_OQO/"
successes_pos = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ]
    )

    successes_pos.append({
        "motif": motif,
        "effector": "OQO",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos = pd.DataFrame(successes_pos)

df_pos

,motif,effector,task,success_count
0,1bcf,OQO,onemotif_twostates_pos,0
1,1prw,OQO,onemotif_twostates_pos,0
2,1qjg,OQO,onemotif_twostates_pos,1
3,1ycr,OQO,onemotif_twostates_pos,3
4,2kl8,OQO,onemotif_twostates_pos,5
5,3ixt,OQO,onemotif_twostates_pos,32
6,4jhw,OQO,onemotif_twostates_pos,0
7,4zyp,OQO,onemotif_twostates_pos,5
8,5ius,OQO,onemotif_twostates_pos,0
9,5tpn,OQO,onemotif_twostates_pos,0


### one motif two states neg with zinc modulation

In [9]:
successes_neg_zn = []
out_dir = f"out/onemotif_twostates_neg_zn/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0) &  ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ]
    )
    successes_neg_zn.append({
        "motif": motif,
        "effector": "[Zn+2]",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_zn = pd.DataFrame(successes_neg_zn)
df_neg_zn

,motif,effector,task,success_count
0,1bcf,[Zn+2],onemotif_twostates_neg,0
1,1prw,[Zn+2],onemotif_twostates_neg,0
2,1qjg,[Zn+2],onemotif_twostates_neg,2
3,1ycr,[Zn+2],onemotif_twostates_neg,6
4,2kl8,[Zn+2],onemotif_twostates_neg,0
5,3ixt,[Zn+2],onemotif_twostates_neg,2
6,4jhw,[Zn+2],onemotif_twostates_neg,0
7,4zyp,[Zn+2],onemotif_twostates_neg,0
8,5ius,[Zn+2],onemotif_twostates_neg,0
9,5tpn,[Zn+2],onemotif_twostates_neg,0


### one motif two states pos with zinc modulation

In [10]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_zn/"
successes_pos_zn = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
    )

    successes_pos_zn.append({
        "motif": motif,
        "effector": "[Zn+2]",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_zn = pd.DataFrame(successes_pos_zn)
df_pos_zn

,motif,effector,task,success_count
0,1bcf,[Zn+2],onemotif_twostates_pos,0
1,1prw,[Zn+2],onemotif_twostates_pos,22
2,1qjg,[Zn+2],onemotif_twostates_pos,4
3,1ycr,[Zn+2],onemotif_twostates_pos,1
4,2kl8,[Zn+2],onemotif_twostates_pos,0
5,3ixt,[Zn+2],onemotif_twostates_pos,28
6,4jhw,[Zn+2],onemotif_twostates_pos,0
7,4zyp,[Zn+2],onemotif_twostates_pos,1
8,5ius,[Zn+2],onemotif_twostates_pos,0
9,5tpn,[Zn+2],onemotif_twostates_pos,0


### 

In [11]:
successes_neg_mg = []
out_dir = f"out/onemotif_twostates_neg_mg/"

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)
    
    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states negative allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] <= 1.0) & (agg["motifRMSD_mean_bound"] > 1.0)  & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ]
    )
    
    successes_neg_mg.append({
        "motif": motif,
        "effector":"[Mg+2]",
        "task": "onemotif_twostates_neg",
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_neg_mg = pd.DataFrame(successes_neg_mg)
df_neg_mg


,motif,effector,task,success_count
0,1bcf,[Mg+2],onemotif_twostates_neg,0
1,1prw,[Mg+2],onemotif_twostates_neg,0
2,1qjg,[Mg+2],onemotif_twostates_neg,0
3,1ycr,[Mg+2],onemotif_twostates_neg,1
4,2kl8,[Mg+2],onemotif_twostates_neg,0
5,3ixt,[Mg+2],onemotif_twostates_neg,2
6,4jhw,[Mg+2],onemotif_twostates_neg,0
7,4zyp,[Mg+2],onemotif_twostates_neg,0
8,5tpn,[Mg+2],onemotif_twostates_neg,0
9,5trv_long,[Mg+2],onemotif_twostates_neg,0


In [13]:
# out_dir = f"./out/onemotif_twostates_pos/"
out_dir = f"out/onemotif_twostates_pos_mg/"
successes_pos_mg = []

for motif in sorted(os.listdir(out_dir)):
    motif_out_dir = os.path.join(out_dir, motif)

    agg_path = os.path.join(motif_out_dir, "_aggresults.csv")
    full_path = os.path.join(motif_out_dir, "_fullresults.csv")

    if not (os.path.exists(agg_path) and os.path.exists(full_path)):
        motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
        full, agg = motif_results(motif_out_dir, motif_pdb, motif)
    else:
        full = pd.read_csv(full_path)
        agg = pd.read_csv(agg_path)

    # condition for one motif two states positive allostery
    n_success = len(
        agg[(agg["motifRMSD_mean_unbound"] > 1.0) & (agg["motifRMSD_mean_bound"] <= 1.0) & ((agg["motifRMSD_mean_bound"] - agg["motifRMSD_mean_unbound"]).abs() >= 0.5) ] 
    )

    successes_pos_mg.append({
        "motif": motif,
        "effector": "[Mg+2]",
        "task": "onemotif_twostates_pos", 
        "success_count": n_success
    })

    # plot_scatter_motifrmsd(agg, motif).show()

df_pos_mg = pd.DataFrame(successes_pos_mg)
df_pos_mg

,motif,effector,task,success_count
0,1bcf,[Mg+2],onemotif_twostates_pos,0
1,1prw,[Mg+2],onemotif_twostates_pos,46
2,1qjg,[Mg+2],onemotif_twostates_pos,2
3,1ycr,[Mg+2],onemotif_twostates_pos,2
4,2kl8,[Mg+2],onemotif_twostates_pos,0
5,3ixt,[Mg+2],onemotif_twostates_pos,20
6,4jhw,[Mg+2],onemotif_twostates_pos,0
7,4zyp,[Mg+2],onemotif_twostates_pos,4
8,5ius,[Mg+2],onemotif_twostates_pos,0
9,5tpn,[Mg+2],onemotif_twostates_pos,0


### aggregate all onemotif two states

In [21]:
df_all = pd.concat([df_pos, df_pos_zn,df_pos_mg, df_neg,  df_neg_zn,df_neg_mg], ignore_index=True)
df_all["type"] = df_all["task"].apply(lambda x: "pos" if "pos" in x else "neg")
heatmap_df = df_all.pivot_table(
    index=["effector", "type"],
    columns="motif",
    values="success_count",
    fill_value=0
)

heatmap_df.index = heatmap_df.index.map(lambda x: f"{x[0]} {x[1]}  ")
heatmap_df = heatmap_df.loc[:, (heatmap_df != 0).any(axis=0)]

fig = px.imshow(
    heatmap_df.values,
    x=heatmap_df.columns,
    y=heatmap_df.index,
    color_continuous_scale="Teal",
    aspect="auto",
    text_auto=True
)

fig.update_layout(
    # title="One motif two states successes",
    font=dict( size=16,color="black"),
    xaxis=dict(side="bottom", tickangle=45),   
    yaxis=dict(autorange="reversed"),
    # xaxis_title="Motif",
    yaxis_title="Effector",
    coloraxis_colorbar=dict(title="Count"),
    width=1200,
    height=400,
    margin=dict(l=50, r=50, t=100, b=50)
)

fig.show()

In [22]:
import plotly.express as px

# 1. Transpose so motifs are rows
heatmap_df_T = heatmap_df.T


# 4. Plot with Plotly
fig = px.imshow(
    heatmap_df_T.values,
    x=heatmap_df_T.columns,
    y=heatmap_df_T.index,
    color_continuous_scale="Teal",
    aspect="auto",
    text_auto=True
)

fig.update_layout(
    font=dict(size=16, color="black"),
    xaxis=dict(side="top", tickangle=0),   # effectors on top
    yaxis=dict(autorange="reversed"),
    xaxis_title="Effector",
    yaxis_title="Motif",
    coloraxis_colorbar=dict(title="Count"),
    width=1000,
    height=800,
    margin=dict(l=50, r=50, t=100, b=50)
)

fig.show()


In [67]:
# fig.write_image("onemotif_twostates_success.pdf")

### two motifs two states

In [123]:
# out_dir = f"./out/twomotif_twostates/"
# out_dir = f"./out/twomoitf_twostates_mg/"
out_dir = f"./out/twomotif_twostates_twoligands/"
# out_dir = "./out/higherantimotif/twomotif_twostates/"

motifA = "3ixt" # active in state 0 when ligand unbound, inactive in state 1 when ligand bound
motifB = "1ycr" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound
# motifB = "6e6r_long" # inactive in state 0 when ligand unbound, active in state 1 when ligand bound

motifA_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifA}.pdb"
motifB_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motifB}.pdb"

motif_out_dir = os.path.join(out_dir,f"{motifA}_{motifB}")
print(motif_out_dir)

./out/twomotif_twostates_twoligands/3ixt_1ycr


In [124]:
motifA_df_full,motifA_df_agg = motif_results(motif_out_dir,motifA_pdb,motifA)
display(plot_scatter_motifrmsd(motifA_df_agg,motifA))
motifB_df_full,motifB_df_agg = motif_results(motif_out_dir,motifB_pdb,motifB)
plot_scatter_motifrmsd(motifB_df_agg,motifB)


In [125]:
success_A = motifA_df_agg[(motifA_df_agg["motifRMSD_mean_unbound"] <= 1.0) & (motifA_df_agg["motifRMSD_mean_bound"] > 1.0) & ((motifA_df_agg["motifRMSD_mean_bound"] - motifA_df_agg["motifRMSD_mean_unbound"]).abs() >= 0.5)] # motif A active in state 0, inactive in state 1
success_B = motifB_df_agg[(motifB_df_agg["motifRMSD_mean_unbound"] > 1.0) & (motifB_df_agg["motifRMSD_mean_bound"] <= 1.0)& ((motifB_df_agg["motifRMSD_mean_bound"] - motifB_df_agg["motifRMSD_mean_unbound"]).abs() >= 0.5)] # motif B inactive in state 0, active in state 1
display(success_A)
display(success_B)

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
19,design26,0.732228,2.289161,0.428667,0.152670,0.537747,0.667688,0.386718,0.533921
31,design37,0.627315,8.494970,0.066834,0.158828,0.802445,0.882553,0.820833,0.828194
35,design40,0.591126,3.585539,0.056860,0.332936,0.642317,0.775785,0.639481,0.762727
44,design49,0.538781,4.211472,0.045860,1.322111,0.558503,0.592559,0.510093,0.496486
45,design5,0.846695,2.042007,0.104683,1.642993,0.466070,0.499519,0.353565,0.336049
46,design50,0.936779,1.803149,0.057379,0.063803,0.670047,0.783040,0.602673,0.661657
56,design6,0.672096,5.875584,0.182406,0.261112,0.635796,0.710766,0.589842,0.513068
64,design67,0.453953,1.569655,0.083305,0.208953,0.607552,0.704142,0.653571,0.677850
79,design80,0.594838,2.144167,0.070519,0.476718,0.817179,0.566659,0.816756,0.378212


,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
28,design34,1.421262,0.679071,0.55397,0.07218,0.559379,0.783833,0.393688,0.643424


In [126]:
set(success_A["design"]).intersection(set(success_B["design"]))

set()

In [58]:
motifB_df_full[motifB_df_full["design"]=="design1"]

,design,state,sample,motifrmsd,plddt,ptm
10,design1,0,0,0.727359,0.679814,0.502158
11,design1,0,1,1.741815,0.710819,0.514460
12,design1,0,2,1.646023,0.709005,0.503220
13,design1,0,3,0.902187,0.684233,0.485717
14,design1,0,4,0.769684,0.672425,0.417673
15,design1,1,0,0.658855,0.767230,0.749349
16,design1,1,1,0.693115,0.755146,0.710810
17,design1,1,2,0.750947,0.776669,0.743204
18,design1,1,3,0.777191,0.758175,0.723047
19,design1,1,4,0.765911,0.774413,0.743548


In [119]:
motifA_df_full[motifA_df_full["design"]=="design91"]

,design,state,sample,motifrmsd,plddt,ptm
910,design91,0,0,0.390635,0.684185,0.589732
911,design91,0,1,0.399948,0.688616,0.620299
912,design91,0,2,0.382589,0.688305,0.585404
913,design91,0,3,0.363413,0.688271,0.584492
914,design91,0,4,0.414448,0.694405,0.612431
915,design91,1,0,1.196500,0.781649,0.802171
916,design91,1,1,1.286581,0.789073,0.805061
917,design91,1,2,1.261281,0.795580,0.803629
918,design91,1,3,1.275448,0.787099,0.794050
919,design91,1,4,1.203851,0.783332,0.785155


In [120]:
motifB_df_full[motifB_df_full["design"]=="design91"]

,design,state,sample,motifrmsd,plddt,ptm
910,design91,0,0,1.514383,0.684185,0.589732
911,design91,0,1,1.265376,0.688616,0.620299
912,design91,0,2,1.426285,0.688305,0.585404
913,design91,0,3,1.468707,0.688271,0.584492
914,design91,0,4,1.311683,0.694405,0.612431
915,design91,1,0,1.466845,0.781649,0.802171
916,design91,1,1,0.635651,0.789073,0.805061
917,design91,1,2,0.679535,0.795580,0.803629
918,design91,1,3,0.587565,0.787099,0.794050
919,design91,1,4,0.660044,0.783332,0.785155
